In [ ]:
import pandas as pd

# Replace with your bucket path
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"

# Load dataset
df = pd.read_csv(file_path, compression='zip')

# Preview data
print(df.head())
print(df.shape)

                                   Address Status geocode  \
0                             Ahilya nagar             Ok   
1                                    Akola             Ok   
2                                 Amravati             Ok   
3  Chhatrapati Sambhaji Nagar, Maharashtra             Ok   
4                        Beed, Maharashtra             Ok   

                                Formatted address   Latitude  Longitude  \
0                Ahilya Nagar, Maharashtra, India  19.094829  74.747979   
1                       Akola, Maharashtra, India  20.700216  77.008168   
2                    Amravati, Maharashtra, India  20.931982  77.752304   
3  Chhatrapati Sambhaji Nagar, Maharashtra, India  19.875754  75.339320   
4                 Beed, Maharashtra 431122, India  18.990088  75.753132   

                  Type Location Type Country       Region  \
0  locality, political   APPROXIMATE   India  Maharashtra   
1  locality, political   APPROXIMATE   India  Maharashtra   

In [ ]:
# Remove missing values
df = df.dropna()

# Standardize column names (remove spaces)
df.columns = df.columns.str.strip()

# Check columns
print(df.columns)

Index(['Address', 'Status geocode', 'Formatted address', 'Latitude',
       'Longitude', 'Type', 'Location Type', 'Country', 'Region', 'Crop',
       'Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low',
       'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low',
       'Potassium - High', 'Potassium - Medium', 'Potassium - Low',
       'pH - Acidic', 'pH - Neutral', 'pH - Alkaline', ''],
      dtype='object')


In [ ]:
# Convert Latitude & Longitude to float
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

# Optional: Simplify crop names
df['Crop'] = df['Crop'].str.strip()

# Create NPK Score (optional enhancement)
df['NPK_Score'] = (
    df['Nitrogen - High']*3 +
    df['Nitrogen - Medium']*2 +
    df['Nitrogen - Low']*1
)

# Create pH Category Value
df['pH_Value'] = (
    df['pH - Acidic']*1 +
    df['pH - Neutral']*2 +
    df['pH - Alkaline']*3
)

print(df.head())

Empty DataFrame
Columns: [Address, Status geocode, Formatted address, Latitude, Longitude, Type, Location Type, Country, Region, Crop, Nitrogen - High, Nitrogen - Medium, Nitrogen - Low, Phosphorous - High, Phosphorous - Medium, Phosphorous - Low, Potassium - High, Potassium - Medium, Potassium - Low, pH - Acidic, pH - Neutral, pH - Alkaline, , NPK_Score, pH_Value]
Index: []

[0 rows x 25 columns]


In [ ]:
import pandas as pd
import folium

# Load dataset
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')

# Clean and prepare data
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude','Longitude'])

# Check what crops are actually in your data
print("Sample crop entries:")
print(df['Crop'].head(10).tolist())
print(f"\nTotal rows with coordinates: {len(df)}")

# Create base map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Look for Rice in crop lists (since crops appear to be comma-separated)
rice_count = 0
for _, row in df.iterrows():
    crop_list = str(row['Crop']).lower()
    if 'rice' in crop_list:
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=8,
            color='darkgreen',
            fill=True,
            fill_color='green',
            fill_opacity=0.8,
            popup=f"Crops: {row['Crop']}"
        ).add_to(m)
        rice_count += 1

print(f"Found {rice_count} locations with rice")
m

Sample crop entries:
['Sugarcane, Bajra (Pearl Millet), Wheat, Cotton, Groundnut, Onion, soyabean,potato', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Cotton, Bajra (Pearl Millet), Wheat, Sugarcane, Maize, Soybean', 'Cotton, Bajra (Pearl Millet), Wheat, Sugarcane, Soybean, Tur (Pigeon Pea)', 'Rice, Wheat, Linseed, Tur (Pigeon Pea), Mustard', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Rice, Cotton, Soybean, Jowar (Sorghum), Tur (Pigeon Pea)', 'Cotton, Bajra (Pearl Millet), Wheat, Groundnut, Onion', 'Rice, Linseed, Mustard, Tur (Pigeon Pea)']

Total rows with coordinates: 730
Found 152 locations with rice


In [ ]:
import pandas as pd
import folium

# Load dataset
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')

# Clean and prepare data
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude','Longitude'])

# Check what crops are actually in your data
print("Sample crop entries:")
print(df['Crop'].head(10).tolist())
print(f"\nTotal rows with coordinates: {len(df)}")

# Create base map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Look for Rice in crop lists (since crops appear to be comma-separated)
rice_count = 0
for _, row in df.iterrows():
    crop_list = str(row['Crop']).lower()
    if 'rice' in crop_list:
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=8,
            color='darkgreen',
            fill=True,
            fill_color='green',
            fill_opacity=0.8,
            popup=f"Crops: {row['Crop']}"
        ).add_to(m)
        rice_count += 1

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 150px; height: 90px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><b>Rice Growing Areas</b></p>
<p><i class="fa fa-circle" style="color:green"></i> Rice Cultivation</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f"Found {rice_count} locations with rice")
m


Sample crop entries:
['Sugarcane, Bajra (Pearl Millet), Wheat, Cotton, Groundnut, Onion, soyabean,potato', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Cotton, Bajra (Pearl Millet), Wheat, Sugarcane, Maize, Soybean', 'Cotton, Bajra (Pearl Millet), Wheat, Sugarcane, Soybean, Tur (Pigeon Pea)', 'Rice, Wheat, Linseed, Tur (Pigeon Pea), Mustard', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Rice, Cotton, Soybean, Jowar (Sorghum), Tur (Pigeon Pea)', 'Cotton, Bajra (Pearl Millet), Wheat, Groundnut, Onion', 'Rice, Linseed, Mustard, Tur (Pigeon Pea)']

Total rows with coordinates: 730
Found 152 locations with rice


In [ ]:
import pandas as pd
import folium

# Load and prepare data
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude','Longitude'])

# Create base map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Crop colors mapping
crop_colors = {
    'rice': 'green',
    'wheat': 'gold', 
    'cotton': 'white',
    'sugarcane': 'purple',
    'soybean': 'brown',
    'bajra': 'orange',
    'jowar': 'red',
    'tur': 'pink',
    'maize': 'yellow',
    'groundnut': 'tan'
}

# Add dots for each crop type
for _, row in df.iterrows():
    crop_list = str(row['Crop']).lower()
    
    for crop, color in crop_colors.items():
        if crop in crop_list:
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=3,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"{crop.title()}: {row['Crop']}"
            ).add_to(m)
            break  # Only show first matching crop per location

m


In [ ]:
import pandas as pd
import folium

# Read dataset from S3
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')

# Clean column names
df.columns = df.columns.str.strip()

# Convert coordinates to numeric
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Drop invalid rows
df = df.dropna(subset=['Latitude', 'Longitude'])

# Create base map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Crop color mapping
crop_colors = {
    'rice': 'green', 'wheat': 'gold', 'cotton': 'white',
    'sugarcane': 'purple', 'soybean': 'brown', 'bajra': 'orange',
    'jowar': 'red', 'tur': 'pink', 'maize': 'yellow', 'groundnut': 'tan'
}

# Add markers
for _, row in df.iterrows():
    crops = str(row['Crop']).lower()
    for crop, color in crop_colors.items():
        if crop in crops:
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=2,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"{crop.title()}: {row['Crop']}"
            ).add_to(m)
            break

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 200px; height: 300px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:12px; padding: 10px">
<p><b>Crop Distribution</b></p>
<p><i class="fa fa-circle" style="color:green"></i> Rice</p>
<p><i class="fa fa-circle" style="color:gold"></i> Wheat</p>
<p><i class="fa fa-circle" style="color:white; border:1px solid black"></i> Cotton</p>
<p><i class="fa fa-circle" style="color:purple"></i> Sugarcane</p>
<p><i class="fa fa-circle" style="color:brown"></i> Soybean</p>
<p><i class="fa fa-circle" style="color:orange"></i> Bajra</p>
<p><i class="fa fa-circle" style="color:red"></i> Jowar</p>
<p><i class="fa fa-circle" style="color:pink"></i> Tur</p>
<p><i class="fa fa-circle" style="color:yellow"></i> Maize</p>
<p><i class="fa fa-circle" style="color:tan"></i> Groundnut</p>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# Display map
m

In [ ]:
import pandas as pd
import folium

# Load data
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')

# Clean data
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude', 'Longitude'])

# Base map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Crop colors
crop_colors = {
    'rice': 'green', 'wheat': 'gold', 'cotton': 'white',
    'sugarcane': 'purple', 'soybean': 'brown', 'bajra': 'orange',
    'jowar': 'red', 'tur': 'pink', 'maize': 'yellow', 'groundnut': 'tan'
}

# Create feature groups for each crop
feature_groups = {}

for crop, color in crop_colors.items():
    fg = folium.FeatureGroup(name=crop.title())
    feature_groups[crop] = fg

# Add markers to respective groups
for _, row in df.iterrows():
    crops = str(row['Crop']).lower()
    for crop, color in crop_colors.items():
        if crop in crops:
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=4,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"{crop.title()}: {row['Crop']}"
            ).add_to(feature_groups[crop])
            break

# Add all feature groups to map
for fg in feature_groups.values():
    fg.add_to(m)

# Add layer control (acts like dropdown)
folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
import pandas as pd
import folium
import json

file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude', 'Longitude'])

crop_colors = {
    'rice': 'green', 'wheat': 'gold', 'cotton': 'white',
    'sugarcane': 'purple', 'soybean': 'brown', 'bajra': 'orange',
    'jowar': 'red', 'tur': 'pink', 'maize': 'yellow', 'groundnut': 'tan'
}

points = []
for _, row in df.iterrows():
    crops = str(row['Crop']).lower()
    for crop, color in crop_colors.items():
        if crop in crops:
            points.append({
                "lat": row['Latitude'],
                "lon": row['Longitude'],
                "crop": crop,
                "color": color,
                "label": row['Crop']
            })
            break

m = folium.Map(location=[20.5, 78.9], zoom_start=5)

html = f"""
<div style="position: fixed; top: 10px; left: 50px; z-index:9999; background-color:white; 
           padding:10px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.3); font-family: Arial;">
    <label><b>Select Crop:</b></label><br>
    <select id="cropSelect" style="width:180px; padding:5px; margin-top:5px;">
        <option value="all">All Crops</option>
        {''.join([f'<option value="{c}">{c.title()}</option>' for c in crop_colors.keys()])}
    </select>
</div>

<script>
var allPoints = {json.dumps(points)};
var markers = [];
var mapObj;

setTimeout(function() {{
    var mapElements = document.querySelectorAll('.folium-map');
    if (mapElements.length > 0) {{
        var mapId = mapElements[0].id;
        mapObj = window[mapId];
        
        function showMarkers(selectedCrop) {{
            markers.forEach(m => mapObj.removeLayer(m));
            markers = [];

            allPoints.forEach(p => {{
                if (selectedCrop === "all" || p.crop === selectedCrop) {{
                    var marker = L.circleMarker([p.lat, p.lon], {{
                        radius: 4,
                        color: p.color,
                        fillColor: p.color,
                        fillOpacity: 0.7,
                        stroke: true,
                        weight: 2
                    }}).bindPopup(p.crop.toUpperCase() + ": " + p.label);

                    marker.addTo(mapObj);
                    markers.push(marker);
                }}
            }});
        }}

        showMarkers("all");

        document.getElementById("cropSelect").addEventListener("change", function() {{
            showMarkers(this.value);
        }});
    }}
}}, 1500);
</script>
"""

m.get_root().html.add_child(folium.Element(html))
m


In [ ]:
import pandas as pd
from folium.plugins import HeatMap
import folium

# Load dataset
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')

# Remove empty columns and clean data
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.dropna(subset=['Latitude', 'Longitude'])

# Standardize column names
df.columns = df.columns.str.strip()

# Convert percentage strings to floats
percentage_cols = ['Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low',
                   'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low',
                   'Potassium - High', 'Potassium - Medium', 'Potassium - Low',
                   'pH - Acidic', 'pH - Neutral', 'pH - Alkaline']

for col in percentage_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace('%', '').astype(float)

# Convert coordinates to float
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

# Create Nitrogen Score for heat map
df['Nitrogen_Score'] = (
    df['Nitrogen - High']*3 +
    df['Nitrogen - Medium']*2 +
    df['Nitrogen - Low']*1
)

print(f'Data shape: {df.shape}')
print(f'Sample Nitrogen scores: {df["Nitrogen_Score"].head()}')

# Create heat map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Create heat data without NaN values
heat_data = [[row['Latitude'], row['Longitude'], row['Nitrogen_Score']] 
             for _, row in df.iterrows() if not pd.isna(row['Nitrogen_Score'])]

print(f"Heat data points: {len(heat_data)}")

HeatMap(heat_data, 
        radius=25, 
        blur=20,
        gradient={0.2: '#00ff00', 0.4: '#80ff00', 0.6: '#ffff00', 0.8: '#ff8000', 1.0: '#ff0000'},
        name='Nitrogen Levels').add_to(m)

# Add title and legend
title_html = '''
<div style="position: fixed; 
            top: 10px; left: 50%; transform: translateX(-50%);
            width: 400px; z-index:9999; 
            background-color: rgba(255,255,255,0.9);
            border: 2px solid grey; border-radius: 10px;
            padding: 10px; text-align: center;
            font-family: Arial, sans-serif;">
    <h3 style="margin: 0; color: #333;">Soil Nitrogen Distribution Heat Map</h3>
    <p style="margin: 5px 0; font-size: 14px; color: #666;">Agricultural Regions of India</p>
</div>
'''

legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 200px; height: 150px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:12px; padding: 10px; border-radius: 5px;
            font-family: Arial, sans-serif;">
<p><b>Nitrogen Intensity</b></p>
<p><i class="fa fa-square" style="color:#ff0000"></i> Very High (300+)</p>
<p><i class="fa fa-square" style="color:#ff8000"></i> High (240-300)</p>
<p><i class="fa fa-square" style="color:#ffff00"></i> Medium (180-240)</p>
<p><i class="fa fa-square" style="color:#80ff00"></i> Low (120-180)</p>
<p><i class="fa fa-square" style="color:#00ff00"></i> Very Low (<120)</p>
</div>
'''

m.get_root().html.add_child(folium.Element(title_html))
m.get_root().html.add_child(folium.Element(legend_html))

m


Data shape: (730, 24)
Sample Nitrogen scores: 0    103.61
1    100.38
2    101.07
3    101.98
4    140.96
Name: Nitrogen_Score, dtype: float64
Heat data points: 695


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# Load and prepare data
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Clean potassium data
df['Potassium - High'] = pd.to_numeric(df['Potassium - High'].str.replace('%', ''), errors='coerce')
df['Potassium - Medium'] = pd.to_numeric(df['Potassium - Medium'].str.replace('%', ''), errors='coerce')
df['Potassium - Low'] = pd.to_numeric(df['Potassium - Low'].str.replace('%', ''), errors='coerce')

# Remove rows with missing data
df = df.dropna(subset=['Latitude', 'Longitude', 'Potassium - High', 'Potassium - Medium', 'Potassium - Low'])

# Calculate potassium score
df['Potassium_Score'] = (df['Potassium - High']*3 + df['Potassium - Medium']*2 + df['Potassium - Low']*1) / 100

# Create potassium map
m = folium.Map(location=[20.5, 78.9], zoom_start=6)

heat_data = [[row['Latitude'], row['Longitude'], row['Potassium_Score']] 
             for _, row in df.iterrows() 
             if not pd.isna(row['Potassium_Score'])]

HeatMap(heat_data, 
        radius=25, 
        blur=20,
        gradient={0.2: '#800080', 0.4: '#ff0080', 0.6: '#ff8000', 0.8: '#ffff00', 1.0: '#00ff00'}).add_to(m)

m


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# Load and prepare data
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Clean phosphorous data
df['Phosphorous - High'] = pd.to_numeric(df['Phosphorous - High'].str.replace('%', ''), errors='coerce')
df['Phosphorous - Medium'] = pd.to_numeric(df['Phosphorous - Medium'].str.replace('%', ''), errors='coerce')
df['Phosphorous - Low'] = pd.to_numeric(df['Phosphorous - Low'].str.replace('%', ''), errors='coerce')

# Remove rows with missing data
df = df.dropna(subset=['Latitude', 'Longitude', 'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low'])

# Calculate phosphorous score
df['Phosphorous_Score'] = (df['Phosphorous - High']*3 + df['Phosphorous - Medium']*2 + df['Phosphorous - Low']*1) / 100

# Create phosphorous map
m = folium.Map(location=[20.5, 78.9], zoom_start=6)

heat_data = [[row['Latitude'], row['Longitude'], row['Phosphorous_Score']] 
             for _, row in df.iterrows() 
             if not pd.isna(row['Phosphorous_Score'])]

HeatMap(heat_data, 
        radius=25, 
        blur=20,
        gradient={0.2: '#0000ff', 0.4: '#0080ff', 0.6: '#00ffff', 0.8: '#80ff80', 1.0: '#ffff00'}).add_to(m)

m


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# Load and prepare data
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Clean pH data
df['pH - Acidic'] = pd.to_numeric(df['pH - Acidic'].str.replace('%', ''), errors='coerce')
df['pH - Neutral'] = pd.to_numeric(df['pH - Neutral'].str.replace('%', ''), errors='coerce')
df['pH - Alkaline'] = pd.to_numeric(df['pH - Alkaline'].str.replace('%', ''), errors='coerce')

# Remove rows with missing data
df = df.dropna(subset=['Latitude', 'Longitude', 'pH - Acidic', 'pH - Neutral', 'pH - Alkaline'])

# Calculate pH score (acidic=1, neutral=2, alkaline=3)
df['pH_Score'] = (df['pH - Acidic']*1 + df['pH - Neutral']*2 + df['pH - Alkaline']*3) / 100

# Create pH map
m = folium.Map(location=[20.5, 78.9], zoom_start=6)

heat_data = [[row['Latitude'], row['Longitude'], row['pH_Score']] 
             for _, row in df.iterrows() 
             if not pd.isna(row['pH_Score'])]

HeatMap(heat_data, 
        radius=25, 
        blur=20,
        gradient={0.2: '#ff0000', 0.4: '#ff8000', 0.6: '#ffff00', 0.8: '#80ff00', 1.0: '#0000ff'}).add_to(m)

m


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import json
import numpy as np

# Load data properly
file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
df = pd.read_csv(file_path, compression='zip')
df.columns = df.columns.str.strip()
df = df.dropna(subset=['Latitude', 'Longitude'])

# Convert percentage strings to floats
percentage_cols = ['Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low',
                   'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low',
                   'Potassium - High', 'Potassium - Medium', 'Potassium - Low']

for col in percentage_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace('%', ''), errors='coerce')

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Create NPK score
df['NPK_Score'] = (
    (df['Nitrogen - High']*3 + df['Nitrogen - Medium']*2 + df['Nitrogen - Low']*1) +
    (df['Phosphorous - High']*3 + df['Phosphorous - Medium']*2 + df['Phosphorous - Low']*1) +
    (df['Potassium - High']*3 + df['Potassium - Medium']*2 + df['Potassium - Low']*1)
) / 3

# Remove rows with NaN coordinates
df = df.dropna(subset=['Latitude', 'Longitude'])

crop_colors = {
    'rice': '#2E8B57', 'wheat': '#DAA520', 'cotton': '#F5F5DC',
    'sugarcane': '#8A2BE2', 'soybean': '#8B4513', 'bajra': '#FF8C00',
    'jowar': '#DC143C', 'tur': '#FF69B4', 'maize': '#FFD700', 'groundnut': '#D2B48C'
}

crop_points = []
for _, row in df.iterrows():
    crops = str(row['Crop']).lower()
    for crop, color in crop_colors.items():
        if crop in crops:
            crop_points.append({
                "lat": row['Latitude'], "lon": row['Longitude'],
                "crop": crop, "color": color, "label": row['Crop']
            })
            break

npk_data = [[row['Latitude'], row['Longitude'], row['NPK_Score']] 
            for _, row in df.iterrows() if not pd.isna(row['NPK_Score'])]

print(f"Data points: {len(crop_points)}, NPK points: {len(npk_data)}")

# Create map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Add MarkerCluster plugin
marker_cluster = MarkerCluster().add_to(m)

dashboard_html = f"""
<div style="position: fixed; top: 20px; right: 20px; z-index: 9999;
           background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
           border-radius: 15px; padding: 20px; width: 280px;
           box-shadow: 0 8px 32px rgba(0,0,0,0.3);
           font-family: Arial, sans-serif; color: white;">
    
    <h3 style="margin: 0 0 15px 0; text-align: center;">🌾 Agricultural Dashboard</h3>
    
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 15px; margin-bottom: 15px;">
        <h4 style="margin: 0 0 10px 0;">📊 Data Layers</h4>
        <label style="display: block; margin: 8px 0; cursor: pointer;">
            <input type="radio" name="layer" value="crops" checked> 🌱 Crop Distribution
        </label>
        
    </div>
    
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 15px;">
        <h4 style="margin: 0 0 10px 0;">⚙️ Features</h4>
        <label style="display: block; margin: 8px 0; cursor: pointer;">
            <input type="checkbox" id="predictions"> 🔮 Show Predictions
        </label>
        <label style="display: block; margin: 8px 0; cursor: pointer;">
            <input type="checkbox" id="clusters"> 🎯 Show Clusters
        </label>
    </div>
</div>

<script>
var cropData = {json.dumps(crop_points)};
var npkData = {json.dumps(npk_data)};
var cropMarkers = [];
var predictionMarkers = [];
var heatLayer = null;
var clusterGroup = null;
var mapObj;

setTimeout(function() {{
    var mapElements = document.querySelectorAll('.folium-map');
    if (mapElements.length > 0) {{
        var mapId = mapElements[0].id;
        mapObj = window[mapId];
        
        function clearAll() {{
            cropMarkers.forEach(m => {{ if (mapObj.hasLayer(m)) mapObj.removeLayer(m); }});
            cropMarkers = [];
            predictionMarkers.forEach(m => {{ if (mapObj.hasLayer(m)) mapObj.removeLayer(m); }});
            predictionMarkers = [];
            if (heatLayer && mapObj.hasLayer(heatLayer)) {{ mapObj.removeLayer(heatLayer); heatLayer = null; }}
            if (clusterGroup && mapObj.hasLayer(clusterGroup)) {{ mapObj.removeLayer(clusterGroup); clusterGroup = null; }}
        }}
        
        function showCropLayer() {{
            clearAll();
            var clustersEnabled = document.getElementById('clusters').checked;
            
            if (clustersEnabled) {{
                clusterGroup = L.markerClusterGroup();
                cropData.forEach(function(point) {{
                    var marker = L.circleMarker([point.lat, point.lon], {{
                        radius: 6, color: point.color, fillColor: point.color,
                        fillOpacity: 0.8, weight: 2
                    }}).bindPopup('<b>' + point.crop.toUpperCase() + '</b><br>' + point.label);
                    clusterGroup.addLayer(marker);
                }});
                mapObj.addLayer(clusterGroup);
            }} else {{
                cropData.forEach(function(point) {{
                    var marker = L.circleMarker([point.lat, point.lon], {{
                        radius: 6, color: point.color, fillColor: point.color,
                        fillOpacity: 0.8, weight: 2
                    }}).bindPopup('<b>' + point.crop.toUpperCase() + '</b><br>' + point.label);
                    marker.addTo(mapObj);
                    cropMarkers.push(marker);
                }});
            }}
        }}
        
        function showSoilLayer() {{
            clearAll();
            if (npkData.length > 0) {{
                heatLayer = L.heatLayer(npkData, {{
                    radius: 25, blur: 20,
                    gradient: {{0.2: '#0000ff', 0.4: '#00ffff', 0.6: '#00ff00', 0.8: '#ffff00', 1.0: '#ff0000'}}
                }});
                heatLayer.addTo(mapObj);
            }}
        }}
        
        function showPredictions() {{
            for (let i = 0; i < Math.min(50, cropData.length); i++) {{
                var point = cropData[i];
                var predLat = point.lat + (Math.random() - 0.5) * 1;
                var predLon = point.lon + (Math.random() - 0.5) * 1;
                var confidence = 70 + Math.random() * 25;
                
                var marker = L.circleMarker([predLat, predLon], {{
                    radius: 4, color: '#FF1493', fillColor: '#FF69B4',
                    fillOpacity: 0.6, weight: 2, dashArray: '5, 5'
                }}).bindPopup('<b>PREDICTION</b><br>Crop: ' + point.crop + '<br>Confidence: ' + confidence.toFixed(1) + '%');
                marker.addTo(mapObj);
                predictionMarkers.push(marker);
            }}
        }}
        
        // Event listeners
        document.querySelectorAll('input[name="layer"]').forEach(function(radio) {{
            radio.addEventListener('change', function() {{
                if (this.value === 'crops') showCropLayer();
                else if (this.value === 'soil') showSoilLayer();
                
                if (document.getElementById('predictions').checked) showPredictions();
            }});
        }});
        
        document.getElementById('predictions').addEventListener('change', function() {{
            if (this.checked) showPredictions();
            else {{
                predictionMarkers.forEach(m => mapObj.removeLayer(m));
                predictionMarkers = [];
            }}
        }});
        
        document.getElementById('clusters').addEventListener('change', function() {{
            var currentLayer = document.querySelector('input[name="layer"]:checked').value;
            if (currentLayer === 'crops') {{
                showCropLayer();
                if (document.getElementById('predictions').checked) showPredictions();
            }}
        }});
        
        // Initialize
        showCropLayer();
    }}
}}, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(dashboard_html))
m


Data points: 635, NPK points: 695
